## Notebook for testing our pipeline for the Generative Recommender

In [2]:
# Imports here
from DataHandler import get_article_feature_string_list
from embedder import Embedder
from rqvae import RQVAE
import torch
import numpy as np
import pickle
import os
import torch, numpy as np, random
import pandas as pd
from transformer import get_all_unique_sids, prepare_dataset, train_model, recommended_next_sid, is_model_trained
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments


# This is ONLY for our tests! Do not use for our full model
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

/opt/anaconda3/envs/devenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### First, get the feature strings from DataHandler

In [3]:
feature_strings = get_article_feature_string_list()
# feature_strings is a list of lists, use list comprehension to make it a list of strings (first 1000 items)
feature_strings = [item[0] for item in feature_strings]
item_ids = [feature_string.split()[0] for feature_string in feature_strings]

print(f'Number of items: {len(feature_strings)}')
print(feature_strings[0])


Number of items: 105542
108775015 Solid Dark Black Strap top Jersey top with narrow shoulder straps. Vest top Garment Upper body Jersey Basic Ladieswear Ladieswear Womens Everyday Basics Jersey Basic


### Next, we use the embedder to get the embeddings that will serve as input for RQ-VAE

In [4]:
if os.path.exists('SBERT_embeddings_fulldata.npy'):
    embeddings = np.load('SBERT_embeddings_fulldata.npy')
    print(f'The embeddings have the shape: {embeddings.shape}')
else:
    embedder = Embedder("sbert")
    embeddings = embedder.encode(feature_strings) # embeddings has shape (n, d), where d = 384 for SBERT
    print(f'The embeddings have the shape: {embeddings.shape}')
    # print(embeddings[0])
    np.save('SBERT_embeddings_fulldata.npy', embeddings)

The embeddings have the shape: (105542, 384)


### Now, we can use the embeddings with RQ-VAE. The code loads hashmaps between item IDs and semantic IDs if they exist. If not, it uses the model to generate semantic IDs and creates (and saves) the hashmaps

In [5]:
input_dim = embeddings.shape[1]
latent_dim = 32
hidden_dim = 256
codebook_size = 512
num_quantizers = 4

set_seed(42)
rqvae = RQVAE(input_dim=input_dim, latent_dim=latent_dim, hidden_dim=hidden_dim, codebook_size=512, num_quantizers=num_quantizers)

# We need to train the model here, before retrieving semantic IDs! Omitting for now... 
# trained_model = ...

print('Generating semantic IDs')

if isinstance(embeddings, np.ndarray):
    embeddings = torch.from_numpy(embeddings).float()
semantic_ids = rqvae.encode_to_semantic_ids(embeddings)
semantic_ids = semantic_ids.detach().numpy() # List of semantic IDs

print(f'semantic_ids has shape: {semantic_ids.shape}')
print('printing the first semantic ID (untrained RQ-VAE)')
print(semantic_ids[0])

item_2_semantic = {}
semantic_2_item = {}

if os.path.exists('item_2_semantic.pkl') and os.path.exists('semantic_2_item.pkl'):
    print('Loading existing hashmaps')

    with open('item_2_semantic.pkl', 'rb') as f:
        item_2_semantic = pickle.load(f)
    with open('semantic_2_item.pkl', 'rb') as f:
        semantic_2_item = pickle.load(f)
    print('Loaded the hashmaps')

else:
    for item_id, semantic_id in zip(item_ids, semantic_ids):
        semantic_tuple = tuple(semantic_id.astype(int))
        item_2_semantic[item_id] = semantic_tuple
        semantic_2_item[semantic_tuple] = item_id

    print(f'Done creating hashmaps. Saving...')
    with open('item_2_semantic.pkl', 'wb') as f:
        pickle.dump(item_2_semantic, f)

    with open('semantic_2_item.pkl', 'wb') as f:
        pickle.dump(semantic_2_item, f)
    print(f'Saved hashmaps to files.')

Generating semantic IDs
semantic_ids has shape: (105542, 4)
printing the first semantic ID (untrained RQ-VAE)
[ 28 450 349 425]
Loading existing hashmaps
Loaded the hashmaps


### Converting transactions data: 
#### user_id: item_id1, ..., item_idn --> user_id: item_SID1, ..., item_SIDn

In [ ]:
transaction = pd.read_pickle('transactions_train.pkl')

customer_sid_sequences = {}
if os.path.exists('customer_sid_sequences.pkl'):
    print('Loading existing customer_sid_sequences')

    with open('customer_sid_sequences.pkl', 'rb') as f:
        customer_sid_sequences = pickle.load(f)
    print('Loaded existing customer_sid_sequences')
else:

    for customer_id, group in transaction.groupby("customer_id"):
        item_ids = group["article_id"].tolist()
        sid_list = [item_2_semantic[str(item_id)] for item_id in item_ids]
        customer_sid_sequences[customer_id] = sid_list
    
    print("customer_sid_sequences has been created, saving...")
    with open("customer_sid_sequences.pkl", "wb") as f:
        pickle.dump(customer_sid_sequences, f)
    print("Saving done!")

first_customer = list(customer_sid_sequences.keys())[0]
print(f"{first_customer}: {customer_sid_sequences[first_customer]}")

Loading existing customer_sid_sequences
Loaded existing customer_sid_sequences
00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657: [(np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(121), np.int64(228), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(137), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(137), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), 

### Taking a subset of customer_sid_sequences for training of transfomer

In [11]:
subset_keys = list(customer_sid_sequences.keys())[:100]
customer_transactions = {k: customer_sid_sequences[k] for k in subset_keys}
print(customer_transactions[list(customer_transactions.keys())[0]])

[(np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(121), np.int64(228), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(137), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(137), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64(450), np.int64(349), np.int64(425)), (np.int64(28), np.int64

### Training transformer with customer_transactions

In [ ]:
window_size=3

if is_model_trained(): 
    print('The model is loaded...')
    model = BartForConditionalGeneration.from_pretrained('./bart-recommender/final_model')
    tokenizer = BartTokenizer.from_pretrained('./bart-recommender/final_model')
else: 
    print('There is no pretrained model, the model will be trained ...')

    unique_sids = get_all_unique_sids(customer_transactions)
    tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')
    tokenizer.add_tokens(unique_sids)

    model = BartForConditionalGeneration.from_pretrained('facebook/bart-base')
    # resize model embeddings after adding token to tokenizer
    model.resize_token_embeddings(len(tokenizer))

    dataset = prepare_dataset(customer_transactions, window_size, tokenizer)
    train_model(dataset, model)

    model.save_pretrained('./bart-recommender/final_model')
    tokenizer.save_pretrained('./bart-recommender/final_model')

There is no pretrained model, the model will be trained ...
['110 375 228 301', '28 121 228 425', '110 411 228 498', '110 450 212 425', '110 450 291 425', '28 411 349 425', '28 121 228 496', '28 450 349 263', '28 411 137 425', '110 375 228 503']


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
/opt/anaconda3/envs/devenv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,3.186800
200,0.254000
300,0.190000
400,0.171400
500,0.182000
600,0.181400
700,0.171300


/opt/anaconda3/envs/devenv/lib/python3.13/site-packages/transformers/modeling_utils.py:4037: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


### Inference: recommend SIDs for given user transaction_sequence

In [ ]:
last_10_customers = list(customer_sid_sequences.keys())[-10:]
test_customers_sequences = {k: customer_sid_sequences[k] for k in last_10_customers}

for customer_id, seqs in test_customers_sequences.items():
    test_history = test_customers_sequences[customer_id][-window_size:]
    test_history = [' '.join(tuple(str(x) for x in sid)) for sid in test_history]
    print(f"test_history: {test_history}")
    recommended_sids = recommended_next_sid(test_history, model, tokenizer, window_size, top_k=12)
    filtered = [ # filter out empty sids
        (sid, semantic_2_item[tuple(int(x) for x in sid.split())])
        for sid in recommended_sids
        if sid.strip() != ''
    ]
    for sid, id in filtered:
        print(f'Rec. SID: {sid} --> article_id: {id}')

test_history: ['110 450 228 503', '28 450 349 425', '28 450 349 425']
Rec. SID: 28 450 349 425 --> article_id: 959461001
Rec. SID: 28 121 228 425 --> article_id: 950449002
Rec. SID: 110 375 228 503 --> article_id: 933909001
Rec. SID: 28 411 137 425 --> article_id: 942451001
Rec. SID: 110 450 228 503 --> article_id: 932578001
Rec. SID: 28 450 137 425 --> article_id: 939503001
Rec. SID: 28 411 349 425 --> article_id: 942451002
Rec. SID: 28 121 228 496 --> article_id: 892560001
Rec. SID: 110 450 212 425 --> article_id: 912862003
Rec. SID: 110 411 228 498 --> article_id: 868018006
Rec. SID: 110 375 228 301 --> article_id: 893955001
test_history: ['28 450 349 425', '28 450 349 425']
Rec. SID: 28 450 349 425 --> article_id: 959461001
Rec. SID: 28 121 228 425 --> article_id: 950449002
Rec. SID: 110 375 228 503 --> article_id: 933909001
Rec. SID: 28 411 137 425 --> article_id: 942451001
Rec. SID: 110 450 228 503 --> article_id: 932578001
Rec. SID: 28 450 137 425 --> article_id: 939503001
Rec. 